# The two-dimensional classical Ising model — student tutorial

Complete the **TODO** functions in order. Each unfinished exercise raises
`NotImplementedError`; run the check and scan cells after filling its dependencies.

Setup, exact reference curves, and plotting helpers are provided.

**Goals**

- Express the partition function as a tensor network.
- Approximate its infinite environment with CTMRG.
- Compute magnetization and specific heat, including automatic differentiation.

We use a square lattice with $J=k_B=1$:

$$
\mathcal H=-\sum_{\langle ij\rangle}\sigma_i\sigma_j-H\sum_i\sigma_i,
\qquad \sigma_i\in\{+1,-1\},\qquad \beta=1/T.
$$

For $N$ sites, the partition function is

$$
Z_N(\beta,H)=\sum_{\{\sigma\}}
\prod_{\langle ij\rangle}e^{\beta\sigma_i\sigma_j}
\prod_i e^{\beta H\sigma_i}.
$$

Each edge carries the Boltzmann matrix $h_{ss'}=e^{\beta ss'}$.
Each vertex carries a copy tensor $x_{s_1s_2s_3s_4}=\delta_{s_1s_2s_3s_4}$
and a local field weight $w_s=e^{\beta Hs}$.

```text
    |     |     |
  --x--h--x--h--x--
    |     |     |
    h     h     h
    |     |     |
  --x--h--x--h--x--
    |     |     |
```

For $\beta\geq0$, split each bond using the symmetric **matrix square root**
$q=h^{1/2}$, with $q q=h$. Absorb four half-bonds into each vertex:

$$
A_{uldr}(\beta,H)=\sum_{s=\pm1}q_{us}q_{ls}q_{sd}q_{sr}\,w_s.
$$

Contracting all copies of $A$ gives $Z_N$. The bond dimension is $D=2$;
we order the legs as **up, left, down, right** (`uldr`).


## 1. Construct the site tensor

Build $A(\beta,H)$ with PyTorch and use float64 throughout.
Keep tensor operations in PyTorch so that we can differentiate them later.

**Exercises 1a–1b:** implement `sqrt_h` and `Ising_A`.
The spin values and copy tensor are provided below.


In [ ]:
# Provided: imports.
import numpy as np
import torch
import matplotlib.pyplot as plt


In [ ]:
sigma = torch.tensor([1.0, -1.0], dtype=torch.float64)
x = torch.zeros((2, 2, 2, 2), dtype=sigma.dtype)
x[0, 0, 0, 0] = x[1, 1, 1, 1] = 1.0


def sqrt_h(beta):
    """Return the symmetric matrix square root of h[s, t] = exp(beta*s*t).

    The ferromagnetic construction assumes beta >= 0. Gradients through
    this square root are used at beta > 0, where h is positive definite.
    """

    # TODO: Exercise 1a.
    # Build the 2x2 Boltzmann matrix h from sigma and beta.
    # Use torch.linalg.eigh to construct its symmetric matrix square root.
    # Check q @ q == h; an elementwise square root is not sufficient.
    raise NotImplementedError("Exercise 1a: implement sqrt_h.")


def Ising_A(beta, H=None):
    """Build the partition-function tensor with legs ordered (up, left, down, right).

    H is the physical magnetic field: the local weight is exp(beta*H*sigma).
    None selects zero field.
    """

    # TODO: Exercise 1b.
    # Obtain the half-bond matrix from sqrt_h(beta).
    # Use local weights exp(beta * H * sigma); treat H=None as zero field.
    # Contract four half-bonds with the copy tensor and field weights.
    # Return shape (2, 2, 2, 2), ordered up, left, down, right.
    # Keep tensor inputs connected to autograd; avoid float(), item(), or NumPy.
    raise NotImplementedError("Exercise 1b: implement Ising_A.")


### 1.1. Check lattice symmetries

Verify that $A$ is unchanged by $90^\circ$ rotations and reflections.
The four permutations below act on its `uldr` legs.

**Checkpoint:** verify both the bond factorization and the lattice symmetries.


In [ ]:
# Provided checks for Exercises 1a–1b.
beta_check = 1.0
q = sqrt_h(beta_check)
h = torch.exp(beta_check * torch.outer(sigma, sigma))
torch.testing.assert_close(q @ q, h)

A = Ising_A(beta_check)
assert A.shape == (2, 2, 2, 2)
for permutation in [(1, 2, 3, 0), (3, 0, 1, 2), (2, 1, 0, 3), (0, 3, 2, 1)]:
    torch.testing.assert_close(A, A.permute(permutation))
print("Bond factorization and square-lattice symmetries verified.")


## 2. Initialize the CTM boundary

Represent the environment by a corner matrix $C$ and an edge tensor $T$.
Square-lattice symmetry lets us use the same tensors in all four directions.

| Tensor | Shape | Meaning |
|---|---|---|
| $C$ | $(b,b)$ | Corner |
| $T$ | $(b,b,D)$ | Edge |

Here $b\leq\chi$ is the current boundary dimension; $\chi$ is the truncation cap.

```text
    C--1           0---T---1
    |                  |
    0                  2
```

Two starting environments:

- **Open boundary:** absorb half-bonds into boundary copy tensors. Start at
  $b=2$ without zero padding; CTMRG grows the dimension up to $\chi$.
- **Random boundary:** use Gram matrices $C=RR^T$ and
  $T[:,:,s]=R_sR_s^T$ to obtain positive-semidefinite boundaries.

For $q=h^{1/2}$, the zero-field open-boundary construction is

```text
    C-- = x--q           --T-- = q--x--q
    |     |                |        |
          q                         q
```

Random boundaries can select either magnetization branch below $T_c$.
We use them in the observable scans.

**Exercise 2b:** implement the random initializer used by the scans.
**Optional 2a:** implement the open-boundary initializer.


In [ ]:
def init_obc_CT(beta):
    """Build zero-field open boundaries in their two-dimensional active space.

    Return C.shape == (2, 2) and T.shape == (2, 2, 2). Subsequent CTMRG
    updates grow the active dimension up to chi without padding zero states.
    """

    # TODO: Exercise 2a (optional).
    # Use sqrt_h(beta) and rank-2 / rank-3 boundary copy tensors.
    # Absorb two half-bonds into C and three into T, following the diagrams.
    # Return shapes (2, 2) and (2, 2, 2), without zero padding.
    raise NotImplementedError("Exercise 2a (optional): implement init_obc_CT.")


def init_rand_CT(A, chi):
    """Build Gram-matrix boundaries: C and each T[:, :, s] are positive semidefinite."""

    # TODO: Exercise 2b.
    # Draw matrices with the same dtype and device as A.
    # Form a Gram matrix for C and one for each physical slice of T.
    # Return C with shape (chi, chi) and T with shape (chi, chi, A.shape[0]).
    raise NotImplementedError("Exercise 2b: implement init_rand_CT.")


## 3. Enlarge, truncate, and absorb

A **corner transfer matrix renormalization group (CTMRG)** step absorbs one
layer, then compresses the boundary back to at most $\chi$ states.

```text
    C---T---T---T---C
    |   |   |   |   |
    T---A---A---A---T
    |   |   |   |   |
    T---A---A---A---T
    |   |   |   |   |
    T---A---A---A---T
    |   |   |   |   |
    C---T---T---T---C
```

### 3.1. Enlarge the corner

Contract one corner, two edges, and one site tensor.
Pair the open legs to form a $(bD)\times(bD)$ matrix $C'$:

```text
    C---T--        C'==
    |   |     =    ||
    T---A--
    |   |
```

**Exercise 3a:** implement `enlarged_C` using the indexed diagram below.


In [ ]:
def enlarged_C(A, C, T):
    r"""
    Build enlarged corner tensor

        C----1 1(0)---T--2(1) = C'==2,7
        0             3(2)      ||
        0(1)          3(0)      4,6
        T--5(2) 5(1)--A--7(3)
        4(0)          6(2)
    """

    # TODO: Exercise 3a.
    # Translate the indexed diagram into a torch.einsum contraction.
    # Return the four open legs in the diagram order: (4, 6, 2, 7).
    # Expected shape: (b, D, b, D). Do not flatten the result yet.
    raise NotImplementedError("Exercise 3a: implement enlarged_C.")


### 3.2. Truncate and update the edge

Diagonalize the real symmetric enlarged corner:

$$
C'=U'\operatorname{diag}(S')U'^T.
$$

Keep $r=\min(\chi,bD)$ eigenvectors with the largest eigenvalue magnitudes.
They form an isometry $U\in\mathbb R^{bD\times r}$, with $U^TU=I_r$.

The approximation $U\operatorname{diag}(S)U^T$ minimizes the Frobenius error
among rank-$r$ matrices. In the retained basis, the new corner is diagonal:

$$
C_{\mathrm{new}}=U^TC'U=\operatorname{diag}(S).
$$

For the edge, absorb one $A$ and compress both enlarged boundary legs using $U$.
The bulk leg remains open:

```text
              |
          ----U----
          |       |             |
          T-------A--    =    T_new--
          |       |             |
          ----U----
              |
```

The indexed diagram in `T_absorb` specifies the leg ordering.

**Exercises 3b–3c:** implement `compress` and `T_absorb`.
Check the retained shapes and the isometry condition.


In [ ]:
def compress(chi, enlarged_C):
    r"""
    Return the retained eigenvalues S and the truncation isometry U.

    The input has shape (b, D, b, D), where b is the current boundary
    dimension. Retain at most min(chi, b*D) eigenvectors,
    ordered by decreasing eigenvalue magnitude. S is a vector, not a matrix.
    """

    # TODO: Exercise 3b.
    # Reshape the enlarged corner into a (b*D, b*D) matrix.
    # Diagonalize it and sort eigenvalues by decreasing absolute value.
    # Keep r = min(chi, b*D) states.
    # Return S with shape (r,) and U with shape (b, D, r).
    # Checkpoint: the flattened U must satisfy U.T @ U == I.
    raise NotImplementedError("Exercise 3b: implement compress.")


def T_absorb(A, T, U):
    r"""
    Absorb one site into the edge tensor and compress with the corner isometry.

        4(2)
        U
        |---------\
        1(0)       3(1)
        1          3(0)      1
        T--2 2(1)--A--6(3) = T--2
        0          5(2)      0
        0          5(1)
        |----------/
        U^\dag
        7(2)
    """

    # TODO: Exercise 3c.
    # Contract T, A, U, and the conjugate of U using the indexed diagram.
    # Leave the new boundary legs and the bulk leg open.
    # Return shape (r, r, D).
    raise NotImplementedError("Exercise 3c: implement T_absorb.")


### 3.3. Compose one CTMRG step

Combine corner enlargement, truncation, and edge absorption.
Store $C_{\mathrm{new}}$ in its diagonal basis.

**Exercise 3d:** combine the three building blocks in `ctm_iter`.


In [ ]:
def ctm_iter(A, C, T, chi):
    r"""
    Perform one CTMRG step, retaining at most chi boundary states.
    """

    # TODO: Exercise 3d.
    # Combine enlarged_C, compress, and T_absorb.
    # Store the retained eigenvalues as a diagonal corner matrix.
    # Return the updated pair (C, T).
    raise NotImplementedError("Exercise 3d: implement ctm_iter.")


## 4. Iterate to a fixed point

After each update:

1. Normalize $C$ and $T$ by their largest absolute entries.
2. Symmetrize the first two legs of $T$ to suppress roundoff errors.
3. Compare consecutive normalized corner spectra, sorted by eigenvalue magnitude.

Stop when the summed absolute spectrum difference is below the tolerance.
If the iteration cap is reached, the environment may still be unconverged.

Scalar normalizations cancel in the observable ratios below.
Finite-$\chi$ errors remain: check larger $\chi$ near the transition.

**Exercise 4:** implement `ctmrg`. Keep its updates differentiable.


In [ ]:
def ctmrg(
    A,
    C,
    T,
    chi,
    ctm_max_iter=100,
    ctm_conv_tol=1.0e-8,
    callback=None,
    raise_on_nonconvergence=False,
):
    """Iterate CTMRG, retaining at most chi states.

    callback(i, residual, finished, A, C, T) monitors each update.
    finished marks convergence or the final allowed iteration.
    Smaller initial boundaries grow up to chi. The stopping test uses no_grad.
    """

    # TODO: Exercise 4.
    # Initialize the normalized corner spectrum from C.
    # Repeat ctm_iter; normalize C and symmetrize and normalize T.
    # Compare spectra sorted by magnitude using the summed absolute difference.
    # Use torch.no_grad only for the stopping test, not for the updates.
    # If dimensions grow, zero-pad spectra only for this comparison.
    # Support callback(i, residual, finished, A, C, T) when provided.
    # Stop at tolerance or the iteration cap; honor raise_on_nonconvergence.
    # Return the final (C, T).
    raise NotImplementedError("Exercise 4: implement ctmrg.")


## 5. Measure the magnetization

Insert a factor of $s$ into the site tensor to obtain the impurity tensor $B$:

$$
B_{uldr}=\sum_{s=\pm1}q_{us}q_{ls}q_{sd}q_{sr}\,w_s\,s.
$$

Contract four corners and four edges into the one-site environment
$\rho_{uldr}$. The magnetization is the normalized contraction

$$
m=\langle\sigma_i\rangle
=\frac{\sum_{uldr}\rho_{uldr}B_{uldr}}
       {\sum_{uldr}\rho_{uldr}A_{uldr}}.
$$

```text
       C---T---C         C---T---C
       |   |   |         |   |   |
  m =  T---B---T    /    T---A---T
       |   |   |         |   |   |
       C---T---C         C---T---C
```

Use the same $\beta$ and $H$ for $A$ and $B$.

**Exercises 5a–5c:** implement the impurity tensor, its environment, and the ratio.


In [ ]:
def f_B(beta, H=None):
    r"""
    Build the magnetization impurity tensor B at the same beta and H as A.
    """

    # TODO: Exercise 5a.
    # Follow the site-tensor construction, with one extra factor of sigma.
    # Use the same field weight and leg order as Ising_A.
    # Return a tensor with shape (2, 2, 2, 2).
    raise NotImplementedError("Exercise 5a: implement f_B.")


def aux_rho_1x1(C, T):
    r"""
    Contract four corners and four edges into rho[up, left, down, right].

        i) C--1 1(0)--T--2(1) = C--T--1
           |          |         |  |
           0          3(2)      0  2

        ii) C--T--1 1(0)--C  =  C--T-----C
            |  |          |     |  |     |
            0  2          |     0  2  3--T
                    4(2)--T              |
                          |              1
                          3(1)

        iii) C------T-------C                C------T------C
             0      |       |                |      |      |
             0(1)   2       |                |      0      |
             T---4(3)    3--T        =       T---1      3--T
             |      5(2)    1                |      2      |
             |      |       1(0)             |      |      |
             C------T-------C                C------T------C
    """

    # TODO: Exercise 5b.
    # Follow the three indexed contractions in the docstring.
    # Contract four corners and four edges, leaving the four site legs open.
    # Return rho in the order up, left, down, right.
    raise NotImplementedError("Exercise 5b: implement aux_rho_1x1.")


def eval_1x1(B, A, rho):
    r"""
    Return the normalized expectation value of B in the environment rho.
    """

    # TODO: Exercise 5c.
    # Contract rho with B for the numerator and with A for the denominator.
    # Return their ratio as a scalar torch tensor.
    raise NotImplementedError("Exercise 5c: implement eval_1x1.")


## 6. Compare with the exact magnetization

At zero field, the critical inverse temperature is

$$
\beta_c=\frac12\log(1+\sqrt2).
$$

The positive spontaneous-magnetization branch is

$$
m_{\mathrm{exact}}(\beta)=
\begin{cases}
0,&\beta\leq\beta_c,\\
\left[1-\sinh^{-4}(2\beta)\right]^{1/8},&\beta>\beta_c.
\end{cases}
$$

- A finite spin-symmetric system has $m=0$ at $H=0$.
- Spontaneous magnetization takes the thermodynamic limit before $H\to0^+$.
- Random boundaries select either sign, so compare $|m|$ with the positive branch.

Use a fixed seed for reproducibility. Check the final convergence residual at
each temperature.

**Provided scan:** run the next two cells after completing Exercises 1–5.


In [ ]:
# Provided: convergence monitor and magnetization scan.
def report_ctm(i, residual, finished, A, C, T):
    """Print the final iteration count and corner-spectrum residual."""
    if finished:
        print(f"iterations={i + 1}, residual={residual.item():.2e}")


torch.manual_seed(7)
chi = 12
betas = np.linspace(0.1, 0.6, 20)
magnetizations = []

with torch.no_grad():
    for beta in betas:
        print(f"beta={beta:.4f}", end=", ")
        A = Ising_A(beta)
        C, T = ctmrg(
            A,
            *init_rand_CT(A, chi),
            chi,
            ctm_max_iter=3000,
            ctm_conv_tol=1.0e-8,
            callback=report_ctm,
        )
        m = eval_1x1(f_B(beta), A, aux_rho_1x1(C, T))
        magnetizations.append(m.item())


In [ ]:
# Provided: exact magnetization and plotting helper.
def exact_magnetization(beta):
    """Return the positive spontaneous magnetization, including the disordered phase."""
    beta = np.asarray(beta, dtype=float)
    beta_c = np.log1p(np.sqrt(2)) / 2
    ordered = beta > beta_c
    result = np.zeros_like(beta)
    result[ordered] = (1 - np.sinh(2 * beta[ordered]) ** (-4)) ** (1 / 8)
    return result


def plot_magnetization(betas, values, title):
    """Compare |m| with the exact zero-field magnetization."""
    values = np.abs(np.asarray(values, dtype=float))
    beta_grid = np.linspace(betas.min(), betas.max(), 200)
    fig, ax = plt.subplots(1, 1, figsize=(5, 4), constrained_layout=True)
    ax.scatter(betas, values, label="CTMRG")
    ax.plot(beta_grid, exact_magnetization(beta_grid), color="black", label="Exact")
    ax.set(xlabel=r"$\beta$", ylabel=r"$|m|$")
    ax.legend()
    fig.suptitle(title)
    return fig, ax


fig_impurity, ax_impurity = plot_magnetization(
    betas, magnetizations, "Magnetization from an impurity tensor"
)
plt.show()


## 7. Compute the partition function per site

The finite thermodynamic quantity is the growth factor per site:

$$
z=\lim_{N\to\infty}Z_N^{1/N},\qquad
\log z=\lim_{N\to\infty}\frac{\log Z_N}{N},\qquad
f=-\frac{\log z}{\beta}.
$$

### 7.1. Contract four patches

A single contraction depends on the normalization of $C$ and $T$.
Form four scalars from the **same converged environment**:

```text
       C---T---C              C-------C
       |   |   |              |       |
  Z1 = T---A---T         Z2 = |       |
       |   |   |              |       |
       C---T---C              C-------C

       C---T---C              C-------C
       |   |   |              |       |
  Z3 = |   |   |         Z4 = T-------T
       |   |   |              |       |
       C---T---C              C-------C
```

Each patch has four corners. $Z_1$ contains four edges and one $A$;
$Z_2$ has no edges; $Z_3$ and $Z_4$ each have two edges.

### 7.2. Isolate one bulk site

Before truncation, let each corner represent a quadrant of linear size $L$:

| Contraction | Patch dimensions |
|---|---|
| $Z_1$ | $(2L+1)\times(2L+1)$ |
| $Z_2$ | $2L\times2L$ |
| $Z_3$ | $(2L+1)\times2L$ |
| $Z_4$ | $2L\times(2L+1)$ |

For large patches, write

$$
\log Z_{n,m}=nm\log z+na+mb+c+\varepsilon_{n,m}.
$$

The combination $\log Z_1+\log Z_2-\log Z_3-\log Z_4$ cancels the boundary
and corner terms. Its net bulk area is one:

$$
(2L+1)^2+(2L)^2-2(2L+1)(2L)=1.
$$

Thus the large-environment estimate is

$$
\boxed{z\simeq\frac{Z_1Z_2}{Z_3Z_4}},\qquad
\log z\simeq\log Z_1+\log Z_2-\log Z_3-\log Z_4.
$$

### 7.3. Cancel normalization factors

Under $C\mapsto cC$ and $T\mapsto tT$,

$$
Z_1\mapsto c^4t^4Z_1,\quad Z_2\mapsto c^4Z_2,\quad
Z_3\mapsto c^4t^2Z_3,\quad Z_4\mapsto c^4t^2Z_4.
$$

All CTM scale factors cancel in the ratio. No normalization history is needed.
The physical scale of $A$ remains: $A\mapsto aA$ gives $z\mapsto az$.

Rotational symmetry gives $Z_3=Z_4$, so compute this contraction once.
Use the logarithmic expression in code and check convergence with $\chi$.

**Exercises 7a–7c:** implement the four contractions, `eval_log_z`, and `eval_z`.


In [ ]:
def partition_contractions(A, C, T):
    """Return Z1, Z2, Z3, Z4 from the four CTM diagrams above.

    Assumes real, reflection-symmetric tensors shared by all directions.
    Consequently the two rectangular contractions satisfy Z3 == Z4.
    """

    # TODO: Exercise 7a.
    # Implement Z1, Z2, Z3, and Z4 using the four patch diagrams.
    # Reuse aux_rho_1x1 for the patch containing A.
    # The empty patch is a closed loop of four corners.
    # Rotational symmetry lets you compute the rectangular patch once.
    # Return four scalar torch tensors.
    raise NotImplementedError("Exercise 7a: implement partition_contractions.")


def eval_log_z(A, C, T):
    """Estimate log(z) = -beta*f from a converged isotropic CTM environment.

    The four scalar contractions are positive for the physical environment.
    Scalar rescalings of C and T cancel; a rescaling of A does not.
    """

    # TODO: Exercise 7b.
    # Call partition_contractions and combine their logarithms.
    # Use the ratio derived above; keep the result connected to autograd.
    # Checkpoint: rescaling C and T must not change this value.
    raise NotImplementedError("Exercise 7b: implement eval_log_z.")


def eval_z(A, C, T):
    """Estimate z = lim Z_N**(1/N), the partition function per site."""

    # TODO: Exercise 7c.
    # Recover z from eval_log_z and return a scalar torch tensor.
    raise NotImplementedError("Exercise 7c: implement eval_z.")


## 8. Observables from automatic differentiation

Differentiate $\log z$ through the full CTMRG calculation.
At $H=0$, magnetization, energy per site, and specific heat are

$$
m=\frac1\beta\frac{\partial\log z}{\partial H},\qquad
u=-\frac{\partial\log z}{\partial\beta},\qquad
c_V=\frac{\partial u}{\partial T}
=\beta^2\frac{\partial^2\log z}{\partial\beta^2}.
$$

Two autograd options matter:

- `retain_graph=True`: reuse the graph for both field and temperature derivatives.
- `create_graph=True`: differentiate the first temperature derivative again.

Use random boundaries to select a magnetization branch.
A tighter CTMRG tolerance is useful for second derivatives.

**Exercise 8:** implement `obs_from_AD`. Return magnetization and specific heat.


In [ ]:
def obs_from_AD(betas, chi, ctm_max_iter=10000, ctm_conv_tol=1e-9):
    """Return zero-field magnetization and specific heat from derivatives of log(z)."""

    # TODO: Exercise 8.
    # For each temperature, create scalar beta and H tensors requiring gradients.
    # Use H=0 and construct A, a random environment, and the converged log(z).
    # Differentiate with respect to H to compute m.
    # Reuse the graph for two derivatives with respect to beta to compute c_V.
    # Use retain_graph and create_graph where needed, as described above.
    # Convert only the final observable values to Python scalars.
    # Return the two lists (m_ad, cv_ad), one entry per beta.
    raise NotImplementedError("Exercise 8: implement obs_from_AD.")


### 8.1. Exact specific heat

Use the zero-field square-lattice formula below. Its logarithmic divergence
marks $\beta_c$; finite $\chi$ limits accuracy near the peak.

Reference: [Caux, Eq. (126)](https://jscaux.org/static/jsc/courses/2019_FPITFT/FROM_PI_TO_FT_2019_notes.pdf).

**Provided reference:** keep this exact formula and plotting helper unchanged.


In [ ]:
# Provided: exact specific heat and plotting helper.
def exact_specific_heat(beta):
    """Exact zero-field specific heat per site for beta > 0 and J = k_B = 1."""
    from scipy.special import ellipk, ellipe

    beta = np.asarray(beta, dtype=float)
    t = np.tanh(2 * beta)
    k = 2 * t / np.cosh(2 * beta)
    K, E = ellipk(k**2), ellipe(k**2)  # SciPy uses the parameter k**2.
    return (
        4
        / np.pi
        * (beta / t) ** 2
        * (K - E - (1 - t**2) * (np.pi / 2 + (2 * t**2 - 1) * K))
    )


def plot_specific_heat(betas, values):
    """Compare CTMRG specific heat with the exact zero-field curve."""
    betas = np.asarray(betas)
    grid = np.linspace(betas.min(), betas.max(), 400)
    fig, ax = plt.subplots(figsize=(6, 4), constrained_layout=True)
    ax.plot(grid, exact_specific_heat(grid), color="black", label="Exact")
    ax.scatter(betas, values, label="CTMRG")
    ax.axvline(
        np.log1p(np.sqrt(2)) / 2, color="gray", linestyle="--", label=r"$\beta_c$"
    )
    ax.set(xlabel=r"$\beta$", ylabel=r"$c_V$", title="Specific heat per site")
    ax.legend()
    return fig, ax


### 8.2. Compare both observables

Sample more densely near $\beta_c$ and compare the AD results with the exact curves.

**Try:** increase $\chi$ and check how the specific-heat peak changes.

**Final checkpoint:** run the scan only after implementing `obs_from_AD`.


In [ ]:
# Run after completing Exercise 8.
torch.manual_seed(7)
betas = np.concatenate(
    [np.linspace(0.3, 0.41, 5), np.linspace(0.43, 0.45, 10), np.linspace(0.48, 0.6, 5)]
)
m_ad, cv_ad = obs_from_AD(betas, chi)


In [ ]:
# Provided: compare both observables with the exact curves.
fig_m, ax_m = plot_magnetization(betas, m_ad, "Magnetization from AD")
fig_heat, ax_heat = plot_specific_heat(betas, cv_ad)

plt.show()
